In [1]:

# from roboflow import Roboflow
# rf = Roboflow(api_key="zgXuwVI2s3j6EXVVrFfq")
# project = rf.workspace("arshs-workspace-radio").project("vzrad2")
# version = project.version(6)
# dataset = version.download("yolov11")
                

In [2]:
import os
import yaml
import shutil
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import random
from collections import defaultdict

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

print("Required libraries imported successfully!")

Required libraries imported successfully!


In [3]:
def filter_mandibular_canal_dataset(source_dir, target_dir, mandibular_canal_class_id=5):
    """
    Filter dataset to include only images that contain Mandibular Canal annotations.
    
    Args:
        source_dir: Path to original dataset
        target_dir: Path to filtered dataset
        mandibular_canal_class_id: Class ID for Mandibular Canal (default: 5)
    """
    
    # Create target directory structure
    os.makedirs(target_dir, exist_ok=True)
    for split in ['train', 'valid', 'test']:
        os.makedirs(f"{target_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{target_dir}/{split}/labels", exist_ok=True)
    
    stats = {
        'train': {'total': 0, 'filtered': 0},
        'valid': {'total': 0, 'filtered': 0},
        'test': {'total': 0, 'filtered': 0}
    }
    
    for split in ['train', 'valid', 'test']:
        source_labels_dir = f"{source_dir}/{split}/labels"
        source_images_dir = f"{source_dir}/{split}/images"
        target_labels_dir = f"{target_dir}/{split}/labels"
        target_images_dir = f"{target_dir}/{split}/images"
        
        if not os.path.exists(source_labels_dir):
            continue
            
        label_files = os.listdir(source_labels_dir)
        stats[split]['total'] = len(label_files)
        
        for label_file in label_files:
            label_path = os.path.join(source_labels_dir, label_file)
            
            # Check if this label file contains Mandibular Canal annotations
            has_mandibular_canal = False
            filtered_annotations = []
            
            try:
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) > 0:
                        class_id = int(parts[0])
                        if class_id == mandibular_canal_class_id:
                            has_mandibular_canal = True
                            # Convert class ID to 0 since we only have one class now
                            filtered_annotations.append(f"0 {' '.join(parts[1:])}\n")
                
                if has_mandibular_canal:
                    # Copy image file
                    image_name = label_file.replace('.txt', '.jpg')
                    if not os.path.exists(os.path.join(source_images_dir, image_name)):
                        # Try .png extension
                        image_name = label_file.replace('.txt', '.png')
                    
                    source_image_path = os.path.join(source_images_dir, image_name)
                    target_image_path = os.path.join(target_images_dir, image_name)
                    
                    if os.path.exists(source_image_path):
                        shutil.copy2(source_image_path, target_image_path)
                        
                        # Save filtered annotations
                        target_label_path = os.path.join(target_labels_dir, label_file)
                        with open(target_label_path, 'w') as f:
                            f.writelines(filtered_annotations)
                        
                        stats[split]['filtered'] += 1
                        
            except Exception as e:
                print(f"Error processing {label_file}: {e}")
    
    return stats

# Filter the dataset
source_dataset_path = "/home/sj/working_dir/3m/vzrad2-6"
filtered_dataset_path = "/home/sj/working_dir/3m/mandibular_canal_dataset"

print("Filtering dataset for Mandibular Canal only...")
filter_stats = filter_mandibular_canal_dataset(source_dataset_path, filtered_dataset_path)

print("\nDataset filtering complete!")
for split, stats in filter_stats.items():
    print(f"{split}: {stats['filtered']}/{stats['total']} images contain Mandibular Canal")

Filtering dataset for Mandibular Canal only...

Dataset filtering complete!
train: 262/4772 images contain Mandibular Canal
valid: 38/2071 images contain Mandibular Canal
test: 21/1580 images contain Mandibular Canal


In [4]:
# Create custom data.yaml for the filtered dataset
def create_custom_yaml(dataset_path):
    """Create a custom data.yaml file for the Mandibular Canal dataset."""
    
    yaml_content = {
        'train': f"{dataset_path}/train/images",
        'val': f"{dataset_path}/valid/images", 
        'test': f"{dataset_path}/test/images",
        'nc': 1,  # Only one class now
        'names': ['Mandibular Canal']
    }
    
    yaml_path = os.path.join(dataset_path, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)
    
    print(f"Custom data.yaml created at: {yaml_path}")
    return yaml_path

# Create the custom YAML file
custom_yaml_path = create_custom_yaml(filtered_dataset_path)

# Display the content of the custom YAML
with open(custom_yaml_path, 'r') as f:
    print("\nCustom data.yaml content:")
    print(f.read())

Custom data.yaml created at: /home/sj/working_dir/3m/mandibular_canal_dataset/data.yaml

Custom data.yaml content:
names:
- Mandibular Canal
nc: 1
test: /home/sj/working_dir/3m/mandibular_canal_dataset/test/images
train: /home/sj/working_dir/3m/mandibular_canal_dataset/train/images
val: /home/sj/working_dir/3m/mandibular_canal_dataset/valid/images



In [5]:
# YOLOv11 Training Configuration and Function
def train_yolov11_segmentation(data_yaml_path, epochs=100, imgsz=640, device='auto'):
    """
    Train YOLOv11 segmentation model for Mandibular Canal detection.
    
    Args:
        data_yaml_path: Path to the custom data.yaml file
        epochs: Number of training epochs
        imgsz: Image size for training
        device: Device to use ('auto', 'cpu', 'cuda', etc.)
    """
    
    # Initialize YOLOv11 segmentation model
    # Using YOLOv11n-seg (nano) for faster training, but you can use 's', 'm', 'l', 'x' for better accuracy
    model = YOLO('yolo11n-seg.pt')  # Load pretrained model
    
    print("Starting YOLOv11 segmentation training...")
    print(f"Dataset: {data_yaml_path}")
    print(f"Epochs: {epochs}")
    print(f"Image size: {imgsz}")
    print(f"Device: {device}")
    
    # Train the model
    results = model.train(
        data=data_yaml_path,
        epochs=epochs,
        imgsz=imgsz,
        device=device,
        project='mandibular_canal_training',
        name='yolo11_mandibular_canal',
        save=True,
        verbose=True,
        patience=20,  # Early stopping patience
        save_period=10,  # Save checkpoint every 10 epochs
        workers=2,  # Reduced for CPU training
        batch=8,  # Reduced batch size for CPU
        optimizer='AdamW',
        lr0=0.01,
        weight_decay=0.0005,
        warmup_epochs=3,
        box=7.5,  # Box loss gain
        cls=0.5,  # Classification loss gain  
        dfl=1.5,  # Distribution focal loss gain
        nbs=64,  # Nominal batch size
        hsv_h=0.015,  # Image HSV-Hue augmentation (fraction)
        hsv_s=0.7,  # Image HSV-Saturation augmentation (fraction)
        hsv_v=0.4,  # Image HSV-Value augmentation (fraction)
        degrees=0.0,  # Image rotation (+/- deg)
        translate=0.1,  # Image translation (+/- fraction)
        scale=0.5,  # Image scale (+/- gain)
        shear=0.0,  # Image shear (+/- deg)
        perspective=0.0,  # Image perspective (+/- fraction)
        flipud=0.0,  # Image flip up-down (probability)
        fliplr=0.5,  # Image flip left-right (probability)
        mosaic=1.0,  # Image mosaic (probability)
        mixup=0.0,  # Image mixup (probability)
        copy_paste=0.0  # Segment copy-paste (probability)
    )
    
    print("\nTraining completed!")
    return model, results

# Start training
print("Initializing YOLOv11 segmentation training...")
trained_model, training_results = train_yolov11_segmentation(
    data_yaml_path=custom_yaml_path,
    epochs=200,  # Adjust as needed
    imgsz=640,
    device='cuda:0'  # Use CPU since CUDA is not available
)

Initializing YOLOv11 segmentation training...


100%|██████████| 5.90M/5.90M [00:00<00:00, 6.36MB/s]


Starting YOLOv11 segmentation training...
Dataset: /home/sj/working_dir/3m/mandibular_canal_dataset/data.yaml
Epochs: 200
Image size: 640
Device: cuda:0
New https://pypi.org/project/ultralytics/8.3.167 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/sj/working_dir/3m/mandibular_canal_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_wi

100%|██████████| 5.35M/5.35M [00:03<00:00, 1.60MB/s]


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2555.6±1438.8 MB/s, size: 52.1 KB)


...

    121/200      2.35G      1.194      1.282     0.7061      1.249         17        640: 100%|██████████| 33/33 [00:02<00:00, 13.12it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:00<00:00, 23.05it/s]

                   all         38         75      0.898      0.933      0.941      0.548      0.864       0.84      0.854      0.262
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 101, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



121 epochs completed in 0.104 hours.
Optimizer stripped from mandibular_canal_training/yolo11_mandibular_canal/weights/last.pt, 6.0MB
Optimizer stripped from mandibular_canal_training/yolo11_mandibular_canal/weights/best.pt, 6.0MB

Validating mandibular_canal_training/yolo11_mandibular_canal/weights/best.pt...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.5.1+cu124 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
YOLO11n-seg summary (fused): 113 layers, 2,834,763 parameters, 0 gradients, 10.2 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:00<00:00, 22.42it/s]


                   all         38         75      0.943      0.933      0.952      0.564      0.876      0.867      0.857      0.292
Speed: 0.1ms preprocess, 0.5ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to mandibular_canal_training/yolo11_mandibular_canal

Training completed!


In [6]:
# Testing and Visualization Functions
def test_model_on_random_image(model, test_images_dir, num_images=3):
    """
    Test the trained model on random images from the dataset.
    
    Args:
        model: Trained YOLO model
        test_images_dir: Directory containing test images
        num_images: Number of random images to test
    """
    
    # Get list of test images
    image_files = [f for f in os.listdir(test_images_dir) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    if not image_files:
        print("No test images found!")
        return
    
    # Select random images
    selected_images = random.sample(image_files, min(num_images, len(image_files)))
    
    print(f"Testing model on {len(selected_images)} random images...")
    
    fig, axes = plt.subplots(2, len(selected_images), figsize=(15, 10))
    if len(selected_images) == 1:
        axes = axes.reshape(2, 1)
    
    for idx, image_file in enumerate(selected_images):
        image_path = os.path.join(test_images_dir, image_file)
        
        # Load and display original image
        original_image = cv2.imread(image_path)
        original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
        
        axes[0, idx].imshow(original_image_rgb)
        axes[0, idx].set_title(f'Original: {image_file}')
        axes[0, idx].axis('off')
        
        # Run inference
        results = model(image_path)
        
        # Plot results
        if len(results) > 0 and results[0].masks is not None:
            # Get the plotted image with predictions
            result_image = results[0].plot(conf=True, line_width=2, font_size=12)
            axes[1, idx].imshow(result_image)
            axes[1, idx].set_title(f'Predicted (Conf > 0.5)')
        else:
            axes[1, idx].imshow(original_image_rgb)
            axes[1, idx].set_title('No detections')
        
        axes[1, idx].axis('off')
        
        # Print detection details
        if len(results) > 0 and results[0].masks is not None:
            boxes = results[0].boxes
            masks = results[0].masks
            if boxes is not None:
                print(f"\n{image_file}:")
                for i, box in enumerate(boxes.data):
                    conf = box[4].item()
                    print(f"  Detection {i+1}: Confidence = {conf:.3f}")
        else:
            print(f"\n{image_file}: No Mandibular Canal detected")
    
    plt.tight_layout()
    plt.show()

def visualize_training_results(results_dir="mandibular_canal_training/yolo11_mandibular_canal"):
    """
    Visualize training results and metrics.
    
    Args:
        results_dir: Directory containing training results
    """
    
    if not os.path.exists(results_dir):
        print(f"Results directory not found: {results_dir}")
        return
    
    # Look for results.png
    results_png = os.path.join(results_dir, "results.png")
    if os.path.exists(results_png):
        print("Training Results:")
        img = cv2.imread(results_png)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(15, 10))
        plt.imshow(img_rgb)
        plt.axis('off')
        plt.title('Training Results')
        plt.show()
    
    # Look for confusion matrix
    confusion_matrix_png = os.path.join(results_dir, "confusion_matrix.png")
    if os.path.exists(confusion_matrix_png):
        print("\nConfusion Matrix:")
        img = cv2.imread(confusion_matrix_png)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 6))
        plt.imshow(img_rgb)
        plt.axis('off')
        plt.title('Confusion Matrix')
        plt.show()

# Test the trained model on random images
test_images_dir = os.path.join(filtered_dataset_path, "test/images")
if os.path.exists(test_images_dir):
    print("Testing trained model on random images...")
    test_model_on_random_image(trained_model, test_images_dir, num_images=3)
else:
    # Use validation images if test images don't exist
    valid_images_dir = os.path.join(filtered_dataset_path, "valid/images")
    if os.path.exists(valid_images_dir):
        print("Testing trained model on random validation images...")
        test_model_on_random_image(trained_model, valid_images_dir, num_images=3)
    else:
        print("No test or validation images found for testing!")

# Visualize training results
print("\nVisualizing training results...")
visualize_training_results()

Testing trained model on random images...
Testing model on 3 random images...

image 1/1 /home/sj/working_dir/3m/mandibular_canal_dataset/test/images/cropped_Manjit-Kaur_2023-10-26181713_1_png.rf.b37d72ef397c4f5f3a10490b24ecd522.jpg: 640x640 2 Mandibular Canals, 3.4ms
Speed: 1.6ms preprocess, 3.4ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

cropped_Manjit-Kaur_2023-10-26181713_1_png.rf.b37d72ef397c4f5f3a10490b24ecd522.jpg:
  Detection 1: Confidence = 0.714
  Detection 2: Confidence = 0.320

image 1/1 /home/sj/working_dir/3m/mandibular_canal_dataset/test/images/cropped_HARJINDER-SINGH_2023-10-20161418_1_png.rf.1aa34057d47630d42486e4b306de888d.jpg: 640x640 2 Mandibular Canals, 3.9ms
Speed: 1.6ms preprocess, 3.9ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

cropped_HARJINDER-SINGH_2023-10-20161418_1_png.rf.1aa34057d47630d42486e4b306de888d.jpg:
  Detection 1: Confidence = 0.768
  Detection 2: Confidence = 0.766

image 1/1 /home/sj/working_dir/3m

<Figure size 1500x1000 with 6 Axes>


Visualizing training results...
Training Results:


<Figure size 1500x1000 with 1 Axes>


Confusion Matrix:


<Figure size 800x600 with 1 Axes>

In [7]:
# Model Evaluation and Saving
import pandas as pd
def evaluate_model(model, data_yaml_path):
    """
    Evaluate the trained model on the validation set.
    
    Args:
        model: Trained YOLO model
        data_yaml_path: Path to the data.yaml file
    """
    
    print("Evaluating model on validation set...")
    
    # Run validation
    validation_results = model.val(
        data=data_yaml_path,
        split='val',
        save_json=True,
        save_hybrid=True,
        conf=0.001,
        iou=0.6,
        max_det=300,
        half=False,
        device='auto'
    )
    
    print("\nValidation Results:")
    print(f"mAP50: {validation_results.box.map50:.4f}")
    print(f"mAP50-95: {validation_results.box.map:.4f}")
    
    if hasattr(validation_results, 'seg'):
        print(f"Segmentation mAP50: {validation_results.seg.map50:.4f}")
        print(f"Segmentation mAP50-95: {validation_results.seg.map:.4f}")
    
    return validation_results

def save_model_info(model, model_path, dataset_info):
    """
    Save model information and training details.
    
    Args:
        model: Trained YOLO model
        model_path: Path to save the model
        dataset_info: Information about the dataset used
    """
    
    # Save the best model
    model.save(model_path)
    print(f"Model saved to: {model_path}")
    
    # Create a summary file
    summary_path = model_path.replace('.pt', '_summary.txt')
    with open(summary_path, 'w') as f:
        f.write("Mandibular Canal Segmentation Model Summary\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Model: YOLOv11 Segmentation\n")
        f.write(f"Task: Mandibular Canal Segmentation\n")
        f.write(f"Classes: 1 (Mandibular Canal)\n")
        f.write(f"Training Date: {pd.Timestamp.now()}\n\n")
        
        f.write("Dataset Information:\n")
        for split, stats in dataset_info.items():
            f.write(f"{split}: {stats['filtered']} images with Mandibular Canal\n")
        
        f.write(f"\nModel file: {model_path}\n")
        f.write(f"Usage: model = YOLO('{model_path}')\n")
        f.write(f"Inference: results = model('path/to/image.jpg')\n")
    
    print(f"Model summary saved to: {summary_path}")

# Evaluate the trained model
validation_results = evaluate_model(trained_model, custom_yaml_path)

# Save the trained model
model_save_path = "/home/sj/working_dir/3m/mandibular_canal_yolo11_best.pt"
save_model_info(trained_model, model_save_path, filter_stats)

print(f"\n{'='*60}")
print("TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"✓ Dataset filtered: {sum(stats['filtered'] for stats in filter_stats.values())} total images with Mandibular Canal")
print(f"✓ Model trained for Mandibular Canal segmentation")
print(f"✓ Model saved to: {model_save_path}")
print(f"✓ Validation mAP50: {validation_results.box.map50:.4f}")
if hasattr(validation_results, 'seg'):
    print(f"✓ Segmentation mAP50: {validation_results.seg.map50:.4f}")
print(f"{'='*60}")

# Usage example
print("\nTo use this model in the future:")
print(f"from ultralytics import YOLO")
print(f"model = YOLO('{model_save_path}')")
print(f"results = model('path/to/new/xray/image.jpg')")
print(f"results[0].show()  # Display results")

Evaluating model on validation set...
WARNING ⚠️ 'save_hybrid' is deprecated and will be removed in in the future.
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.5.1+cu124 CUDA:auto (NVIDIA RTX 6000 Ada Generation, 48640MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3558.8±605.8 MB/s, size: 49.7 KB)


val: Scanning /home/sj/working_dir/3m/mandibular_canal_dataset/valid/labels.cache... 38 images, 0 backgrounds, 0 corrupt: 100%|██████████| 38/38 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:01<00:00,  3.47it/s]


                   all         38         75      0.933      0.947      0.952      0.561      0.863      0.853      0.847      0.275
Speed: 2.3ms preprocess, 5.5ms inference, 0.0ms loss, 2.4ms postprocess per image
Saving mandibular_canal_training/yolo11_mandibular_canal2/predictions.json...
Results saved to mandibular_canal_training/yolo11_mandibular_canal2

Validation Results:
mAP50: 0.9525
mAP50-95: 0.5611
Segmentation mAP50: 0.8473
Segmentation mAP50-95: 0.2754
Model saved to: /home/sj/working_dir/3m/mandibular_canal_yolo11_best.pt
Model summary saved to: /home/sj/working_dir/3m/mandibular_canal_yolo11_best_summary.txt

TRAINING COMPLETE!
✓ Dataset filtered: 321 total images with Mandibular Canal
✓ Model trained for Mandibular Canal segmentation
✓ Model saved to: /home/sj/working_dir/3m/mandibular_canal_yolo11_best.pt
✓ Validation mAP50: 0.9525
✓ Segmentation mAP50: 0.8473

To use this model in the future:
from ultralytics import YOLO
model = YOLO('/home/sj/working_dir/3m/mandibula

In [8]:
# Comprehensive Testing and Evaluation with Comparison Images
import matplotlib.patches as patches
from matplotlib.patches import Polygon
import json

def parse_yolo_segmentation_annotation(label_file, img_width, img_height):
    """
    Parse YOLO segmentation annotation file to get polygon coordinates.
    
    Args:
        label_file: Path to the .txt annotation file
        img_width: Width of the image
        img_height: Height of the image
    
    Returns:
        List of polygon coordinates
    """
    polygons = []
    
    if not os.path.exists(label_file):
        return polygons
        
    with open(label_file, 'r') as f:
        lines = f.readlines()
        
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 7:  # class_id + at least 6 coordinates (3 points)
            class_id = int(parts[0])
            if class_id == 0:  # Our mandibular canal class
                # Convert normalized coordinates to pixel coordinates
                coords = []
                for i in range(1, len(parts), 2):
                    if i + 1 < len(parts):
                        x = float(parts[i]) * img_width
                        y = float(parts[i + 1]) * img_height
                        coords.append([x, y])
                if len(coords) >= 3:  # Valid polygon needs at least 3 points
                    polygons.append(coords)
    
    return polygons

def create_mask_from_polygons(polygons, img_width, img_height):
    """
    Create a binary mask from polygon coordinates.
    
    Args:
        polygons: List of polygon coordinates
        img_width: Width of the image
        img_height: Height of the image
    
    Returns:
        Binary mask as numpy array
    """
    mask = np.zeros((img_height, img_width), dtype=np.uint8)
    
    for polygon in polygons:
        if len(polygon) >= 3:
            # Convert polygon to integer coordinates
            polygon_int = np.array(polygon, dtype=np.int32)
            cv2.fillPoly(mask, [polygon_int], 255)
    
    return mask

def overlay_mask_on_image(image, mask, color=(0, 255, 0), alpha=0.4):
    """
    Overlay a colored mask on an image.
    
    Args:
        image: Original image (BGR format)
        mask: Binary mask
        color: Color for the mask overlay (BGR format)
        alpha: Transparency of the overlay
    
    Returns:
        Image with mask overlay
    """
    overlay = image.copy()
    
    # Create colored mask
    colored_mask = np.zeros_like(image)
    colored_mask[mask > 0] = color
    
    # Blend with original image
    result = cv2.addWeighted(overlay, 1-alpha, colored_mask, alpha, 0)
    
    return result

def comprehensive_model_evaluation():
    """
    Perform comprehensive evaluation of the trained model on all test images.
    """
    
    # Create output directories
    test_eval_dir = "/home/sj/working_dir/3m/test_data_evaluated"
    comparison_dir = "/home/sj/working_dir/3m/comparison_images"
    
    os.makedirs(test_eval_dir, exist_ok=True)
    os.makedirs(comparison_dir, exist_ok=True)
    
    print("Created output directories:")
    print(f"- Test evaluations: {test_eval_dir}")
    print(f"- Comparison images: {comparison_dir}")
    
    # Get test data paths
    test_images_dir = os.path.join(filtered_dataset_path, "test/images")
    test_labels_dir = os.path.join(filtered_dataset_path, "test/labels")
    
    if not os.path.exists(test_images_dir):
        # Fallback to validation data if test data doesn't exist
        test_images_dir = os.path.join(filtered_dataset_path, "valid/images")
        test_labels_dir = os.path.join(filtered_dataset_path, "valid/labels")
        print("Using validation data for testing (test data not found)")
    
    if not os.path.exists(test_images_dir):
        print("No test or validation images found!")
        return
    
    # Get all test images
    image_files = [f for f in os.listdir(test_images_dir) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    print(f"\nFound {len(image_files)} test images")
    print("Starting comprehensive evaluation...")
    
    # Statistics tracking
    total_images = len(image_files)
    images_with_detections = 0
    total_detections = 0
    
    for idx, image_file in enumerate(image_files):
        print(f"Processing {idx+1}/{total_images}: {image_file}")
        
        # Load original image
        image_path = os.path.join(test_images_dir, image_file)
        original_image = cv2.imread(image_path)
        img_height, img_width = original_image.shape[:2]
        
        # Get corresponding label file
        label_file = image_file.replace('.jpg', '.txt').replace('.png', '.txt')
        label_path = os.path.join(test_labels_dir, label_file)
        
        # Parse ground truth annotations
        gt_polygons = parse_yolo_segmentation_annotation(label_path, img_width, img_height)
        gt_mask = create_mask_from_polygons(gt_polygons, img_width, img_height)
        
        # Run model inference
        results = model(image_path)
        
        # Process predictions
        pred_mask = np.zeros((img_height, img_width), dtype=np.uint8)
        detections_count = 0
        
        if len(results) > 0 and results[0].masks is not None:
            masks = results[0].masks
            boxes = results[0].boxes
            
            for i, mask_data in enumerate(masks.data):
                # Convert mask to numpy array and resize to image dimensions
                mask_np = mask_data.cpu().numpy()
                mask_resized = cv2.resize(mask_np, (img_width, img_height))
                mask_binary = (mask_resized > 0.5).astype(np.uint8) * 255
                
                # Combine with existing predictions
                pred_mask = cv2.bitwise_or(pred_mask, mask_binary)
                detections_count += 1
        
        if detections_count > 0:
            images_with_detections += 1
            total_detections += detections_count
        
        # Create overlays
        gt_overlay = overlay_mask_on_image(original_image, gt_mask, color=(0, 255, 0), alpha=0.4)  # Green for GT
        pred_overlay = overlay_mask_on_image(original_image, pred_mask, color=(0, 0, 255), alpha=0.4)  # Red for predictions
        
        # Save prediction result to test_data_evaluated
        pred_result_path = os.path.join(test_eval_dir, f"pred_{image_file}")
        cv2.imwrite(pred_result_path, pred_overlay)
        
        # Create comparison image (side by side)
        comparison_image = np.hstack([gt_overlay, pred_overlay])
        
        # Add text labels
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1
        thickness = 2
        
        # Add "Ground Truth" label
        cv2.putText(comparison_image, "Ground Truth", (10, 30), font, font_scale, (255, 255, 255), thickness)
        # Add "Prediction" label
        cv2.putText(comparison_image, "Prediction", (img_width + 10, 30), font, font_scale, (255, 255, 255), thickness)
        
        # Add detection count info
        gt_count = len(gt_polygons)
        pred_count = detections_count
        cv2.putText(comparison_image, f"GT: {gt_count}", (10, img_height - 10), font, 0.7, (0, 255, 0), 2)
        cv2.putText(comparison_image, f"Pred: {pred_count}", (img_width + 10, img_height - 10), font, 0.7, (0, 0, 255), 2)
        
        # Save comparison image
        comparison_path = os.path.join(comparison_dir, f"comparison_{image_file}")
        cv2.imwrite(comparison_path, comparison_image)
    
    # Print evaluation summary
    print(f"\n{'='*60}")
    print("COMPREHENSIVE EVALUATION COMPLETE!")
    print(f"{'='*60}")
    print(f"Total images processed: {total_images}")
    print(f"Images with detections: {images_with_detections}")
    print(f"Detection rate: {images_with_detections/total_images*100:.1f}%")
    print(f"Total detections: {total_detections}")
    print(f"Average detections per image: {total_detections/total_images:.2f}")
    print(f"\nOutput directories:")
    print(f"- Predictions saved to: {test_eval_dir}")
    print(f"- Comparisons saved to: {comparison_dir}")
    print(f"{'='*60}")

def create_evaluation_summary():
    """
    Create a summary report of the evaluation.
    """
    summary_path = "/home/sj/working_dir/3m/evaluation_summary.txt"
    
    with open(summary_path, 'w') as f:
        f.write("Mandibular Canal Segmentation - Evaluation Summary\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Evaluation Date: {pd.Timestamp.now()}\n")
        f.write(f"Model: YOLOv11 Segmentation (Nano)\n")
        f.write(f"Task: Mandibular Canal Segmentation\n\n")
        
        f.write("Dataset Information:\n")
        for split, stats in filter_stats.items():
            f.write(f"- {split}: {stats['filtered']} images with Mandibular Canal\n")
        
        f.write(f"\nOutput Directories:\n")
        f.write(f"- Predictions: /home/sj/working_dir/3m/test_data_evaluated/\n")
        f.write(f"- Comparisons: /home/sj/working_dir/3m/comparison_images/\n")
        f.write(f"- Model file: /home/sj/working_dir/3m/mandibular_canal_yolo11_best.pt\n")
        
        f.write(f"\nUsage Instructions:\n")
        f.write(f"1. Load model: model = YOLO('/home/sj/working_dir/3m/mandibular_canal_yolo11_best.pt')\n")
        f.write(f"2. Run inference: results = model('path/to/image.jpg')\n")
        f.write(f"3. Display results: results[0].show()\n")
    
    print(f"Evaluation summary saved to: {summary_path}")

# Note: The model needs to be trained first before running this evaluation
print("This evaluation code will run automatically after model training is complete.")
print("Make sure to run all previous cells first to train the model.")

# Check if model exists and run evaluation
if 'trained_model' in locals():
    print("\nTrained model found. Starting comprehensive evaluation...")
    model = trained_model
    comprehensive_model_evaluation()
    create_evaluation_summary()
else:
    print("\nTrained model not found. Please run the training cells first.")
    print("After training, you can run this cell again to perform comprehensive evaluation.")

This evaluation code will run automatically after model training is complete.
Make sure to run all previous cells first to train the model.

Trained model found. Starting comprehensive evaluation...
Created output directories:
- Test evaluations: /home/sj/working_dir/3m/test_data_evaluated
- Comparison images: /home/sj/working_dir/3m/comparison_images

Found 21 test images
Starting comprehensive evaluation...
Processing 1/21: cropped_Satpal-Singh_2023-10-26163335_1_png.rf.b7d11ac98aa9fb355f4c2ee0863218e3.jpg

image 1/1 /home/sj/working_dir/3m/mandibular_canal_dataset/test/images/cropped_Satpal-Singh_2023-10-26163335_1_png.rf.b7d11ac98aa9fb355f4c2ee0863218e3.jpg: 640x640 2 Mandibular Canals, 5.4ms
Speed: 1.7ms preprocess, 5.4ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)
Processing 2/21: cropped_gurtej-singh_2023-10-26184657_1_png.rf.ca6f0f42843c9f0a754008327cd1146c.jpg

image 1/1 /home/sj/working_dir/3m/mandibular_canal_dataset/test/images/cropped_gurtej-singh_2023

In [9]:
def create_comparison_mosaic(comparison_dir, output_path="comparison_mosaic.png", max_width=3000):
    """
    Create a mosaic of all images in the comparison images directory and save as PNG.
    
    Args:
        comparison_dir: Directory containing comparison images
        output_path: Path to save the mosaic PNG
        max_width: Maximum width of the mosaic image
    """
    # Get all image files
    image_files = [f for f in os.listdir(comparison_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if not image_files:
        print("No images found in comparison directory!")
        return

    # Load all images
    images = []
    for fname in image_files:
        img = cv2.imread(os.path.join(comparison_dir, fname))
        if img is not None:
            images.append(img)
    if not images:
        print("No valid images loaded!")
        return

    # Get image shape (assuming all images are same size)
    img_h, img_w = images[0].shape[:2]
    n_images = len(images)

    # Estimate grid size (rows and cols)
    # Try to make the mosaic as square as possible
    cols = int(np.ceil(np.sqrt(n_images)))
    rows = int(np.ceil(n_images / cols))

    # Optionally, resize images if mosaic would be too wide
    mosaic_img_w = cols * img_w
    mosaic_img_h = rows * img_h
    scale = 1.0
    if mosaic_img_w > max_width:
        scale = max_width / mosaic_img_w
        img_w = int(img_w * scale)
        img_h = int(img_h * scale)
        images = [cv2.resize(img, (img_w, img_h)) for img in images]
        mosaic_img_w = cols * img_w
        mosaic_img_h = rows * img_h

    # Create blank mosaic
    mosaic = np.zeros((mosaic_img_h, mosaic_img_w, 3), dtype=np.uint8)

    # Paste images into mosaic
    for idx, img in enumerate(images):
        row = idx // cols
        col = idx % cols
        y0 = row * img_h
        x0 = col * img_w
        mosaic[y0:y0+img_h, x0:x0+img_w] = img

    # Save mosaic
    cv2.imwrite(output_path, mosaic)
    print(f"Mosaic saved to: {output_path}")

# Usage
comparison_dir = "/home/sj/working_dir/3m/comparison_images"
create_comparison_mosaic(comparison_dir)

Mosaic saved to: comparison_mosaic.png


In [12]:
import random
import shutil

def collect_random_images(source_dirs, output_dir, images_per_dir=5):
    """
    Collect random images from multiple source directories and copy to output_dir with dataset suffix.
    
    Args:
        source_dirs: List of tuples (dir_path, dataset_name)
        output_dir: Directory to save selected images
        images_per_dir: Number of images to select from each directory
    """
    os.makedirs(output_dir, exist_ok=True)
    selected_images = []

    for dir_path, dataset_name in source_dirs:
        image_files = [f for f in os.listdir(dir_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        chosen = random.sample(image_files, min(images_per_dir, len(image_files)))
        for fname in chosen:
            src_path = os.path.join(dir_path, fname)
            # Add dataset name as suffix before extension
            base, ext = os.path.splitext(fname)
            new_fname = f"{base}_{dataset_name}{ext}"
            dst_path = os.path.join(output_dir, new_fname)
            shutil.copy2(src_path, dst_path)
            selected_images.append(new_fname)
    print(f"Copied {len(selected_images)} images to {output_dir}")
    return selected_images

def predict_on_images(model, images_dir, save_predictions=True, save_comparisons=True):
    """
    Run prediction using trained YOLO model on all images in images_dir and save results.
    
    Args:
        model: Trained YOLO model
        images_dir: Directory containing images to predict
        save_predictions: Whether to save predicted mask overlays
        save_comparisons: Whether to save comparison images
    """
    # Create output directories
    predictions_dir = os.path.join(os.path.dirname(images_dir), "wild_predictions")
    comparisons_dir = os.path.join(os.path.dirname(images_dir), "wild_comparisons")
    
    if save_predictions:
        os.makedirs(predictions_dir, exist_ok=True)
    if save_comparisons:
        os.makedirs(comparisons_dir, exist_ok=True)
    
    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Running prediction on {len(image_files)} images...")
    
    detection_stats = {
        'total_images': len(image_files),
        'images_with_detections': 0,
        'total_detections': 0
    }
    
    for idx, fname in enumerate(image_files):
        print(f"Processing {idx+1}/{len(image_files)}: {fname}")
        
        img_path = os.path.join(images_dir, fname)
        original_image = cv2.imread(img_path)
        
        if original_image is None:
            print(f"Could not load image: {fname}")
            continue
            
        img_height, img_width = original_image.shape[:2]
        
        # Run model inference
        results = model(img_path)
        
        # Process predictions and create mask overlay
        pred_mask = np.zeros((img_height, img_width), dtype=np.uint8)
        detections_count = 0
        confidence_scores = []
        
        if len(results) > 0 and results[0].masks is not None:
            masks = results[0].masks
            boxes = results[0].boxes
            
            print(f"  Found {len(masks.data)} detections")
            
            for i, mask_data in enumerate(masks.data):
                # Convert mask to numpy array and resize to image dimensions
                mask_np = mask_data.cpu().numpy()
                mask_resized = cv2.resize(mask_np, (img_width, img_height))
                mask_binary = (mask_resized > 0.5).astype(np.uint8) * 255
                
                # Combine with existing predictions
                pred_mask = cv2.bitwise_or(pred_mask, mask_binary)
                detections_count += 1
                
                # Get confidence score
                if boxes is not None and i < len(boxes.data):
                    conf = boxes.data[i][4].item()
                    confidence_scores.append(conf)
                    print(f"    Detection {i+1}: Confidence = {conf:.3f}")
        else:
            print(f"  No Mandibular Canal detected")
        
        if detections_count > 0:
            detection_stats['images_with_detections'] += 1
            detection_stats['total_detections'] += detections_count
        
        # Create prediction overlay (red mask for predictions)
        pred_overlay = overlay_mask_on_image(original_image, pred_mask, color=(0, 0, 255), alpha=0.4)
        
        # Add confidence information to the overlay
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.8
        thickness = 2
        
        # Add detection count and average confidence
        if detections_count > 0:
            avg_conf = np.mean(confidence_scores) if confidence_scores else 0
            text = f"Detections: {detections_count}, Avg Conf: {avg_conf:.3f}"
            cv2.putText(pred_overlay, text, (10, 30), font, font_scale, (255, 255, 255), thickness)
        else:
            cv2.putText(pred_overlay, "No Mandibular Canal Detected", (10, 30), font, font_scale, (255, 255, 255), thickness)
        
        # Save predicted overlay image
        if save_predictions:
            pred_filename = f"pred_{fname}"
            pred_path = os.path.join(predictions_dir, pred_filename)
            cv2.imwrite(pred_path, pred_overlay)
        
        # Create comparison image (original left, prediction right)
        if save_comparisons:
            # Ensure both images have the same dimensions
            original_resized = cv2.resize(original_image, (img_width, img_height))
            pred_resized = cv2.resize(pred_overlay, (img_width, img_height))
            
            # Create side-by-side comparison
            comparison_image = np.hstack([original_resized, pred_resized])
            
            # Add labels
            label_font_scale = 1.0
            label_thickness = 3
            
            # Add "Original" label
            cv2.putText(comparison_image, "Original", (10, 50), font, label_font_scale, (255, 255, 255), label_thickness)
            # Add "Prediction" label
            cv2.putText(comparison_image, "Prediction", (img_width + 10, 50), font, label_font_scale, (255, 255, 255), label_thickness)
            
            # Add dataset information
            dataset_name = fname.split('_')[-1].split('.')[0] if '_' in fname else "unknown"
            cv2.putText(comparison_image, f"Dataset: {dataset_name}", (10, img_height - 10), font, 0.7, (200, 200, 200), 2)
            
            # Save comparison image
            comp_filename = f"comparison_{fname}"
            comp_path = os.path.join(comparisons_dir, comp_filename)
            cv2.imwrite(comp_path, comparison_image)
    
    # Print summary statistics
    print(f"\n{'='*50}")
    print("WILD TEST PREDICTION SUMMARY")
    print(f"{'='*50}")
    print(f"Total images processed: {detection_stats['total_images']}")
    print(f"Images with detections: {detection_stats['images_with_detections']}")
    print(f"Detection rate: {detection_stats['images_with_detections']/detection_stats['total_images']*100:.1f}%")
    print(f"Total detections: {detection_stats['total_detections']}")
    print(f"Average detections per image: {detection_stats['total_detections']/detection_stats['total_images']:.2f}")
    
    if save_predictions:
        print(f"\nPrediction overlays saved to: {predictions_dir}")
    if save_comparisons:
        print(f"Comparison images saved to: {comparisons_dir}")
    print(f"{'='*50}")
    
    return detection_stats

# Define source directories and dataset names
source_dirs = [
    ("/home/sj/working_dir/third_molar_project_cuda0/raw/1.adld/adld/images", "adld"),
    ("/home/sj/working_dir/third_molar_project_cuda0/raw/2.dentex/dentex/xrays", "dentex"),
    ("/home/sj/working_dir/third_molar_project_cuda0/raw/3.tsxk/tsxk/img", "tsxk"),
    ("/home/sj/working_dir/third_molar_project_cuda0/raw/4.tufts/tufts/Radiographs", "tufts"),
    ("/home/sj/working_dir/third_molar_project_cuda0/raw/5.uspforp/uspforp/image", "uspforp")
]

output_dir = "/home/sj/working_dir/3m/wild_test_images"
collect_random_images(source_dirs, output_dir, images_per_dir=5)

# Run prediction using trained_model with enhanced saving capabilities
if 'trained_model' in locals():
    print("Running predictions on wild test images...")
    wild_stats = predict_on_images(trained_model, output_dir, save_predictions=True, save_comparisons=True)
    
    # Create a summary report for wild test results
    wild_summary_path = "/home/sj/working_dir/3m/wild_test_summary.txt"
    with open(wild_summary_path, 'w') as f:
        f.write("Wild Test Images - Mandibular Canal Detection Summary\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Test Date: {pd.Timestamp.now()}\n")
        f.write(f"Model: YOLOv11 Segmentation (Nano)\n")
        f.write(f"Task: Mandibular Canal Detection on External Datasets\n\n")
        
        f.write("Dataset Sources:\n")
        for source_dir, dataset_name in source_dirs:
            f.write(f"- {dataset_name.upper()}: {source_dir}\n")
        
        f.write(f"\nResults:\n")
        f.write(f"- Total images tested: {wild_stats['total_images']}\n")
        f.write(f"- Images with detections: {wild_stats['images_with_detections']}\n")
        f.write(f"- Detection rate: {wild_stats['images_with_detections']/wild_stats['total_images']*100:.1f}%\n")
        f.write(f"- Total detections: {wild_stats['total_detections']}\n")
        f.write(f"- Average detections per image: {wild_stats['total_detections']/wild_stats['total_images']:.2f}\n")
        
        f.write(f"\nOutput Directories:\n")
        f.write(f"- Original test images: {output_dir}\n")
        f.write(f"- Prediction overlays: {os.path.join(os.path.dirname(output_dir), 'wild_predictions')}\n")
        f.write(f"- Comparison images: {os.path.join(os.path.dirname(output_dir), 'wild_comparisons')}\n")
    
    print(f"Wild test summary saved to: {wild_summary_path}")
else:
    print("Trained model not found. Please run the training cells first.")
    print("The prediction function is ready to use once the model is trained.")

Copied 25 images to /home/sj/working_dir/3m/wild_test_images
Running predictions on wild test images...
Running prediction on 25 images...
Processing 1/25: train_324_dentex.png

image 1/1 /home/sj/working_dir/3m/wild_test_images/train_324_dentex.png: 320x640 3 Mandibular Canals, 4.4ms
Speed: 9.6ms preprocess, 4.4ms inference, 1.2ms postprocess per image at shape (1, 3, 320, 640)
  Found 3 detections
    Detection 1: Confidence = 0.640
    Detection 2: Confidence = 0.543
    Detection 3: Confidence = 0.274
Processing 2/25: 545_tsxk.jpg

image 1/1 /home/sj/working_dir/3m/wild_test_images/545_tsxk.jpg: 352x640 1 Mandibular Canal, 4.4ms
Speed: 1.3ms preprocess, 4.4ms inference, 1.4ms postprocess per image at shape (1, 3, 352, 640)
  Found 1 detections
    Detection 1: Confidence = 0.653
Processing 3/25: 237-F-32_uspforp.jpg

image 1/1 /home/sj/working_dir/3m/wild_test_images/237-F-32_uspforp.jpg: 352x640 2 Mandibular Canals, 4.1ms
Speed: 1.5ms preprocess, 4.1ms inference, 1.1ms postprocess

In [13]:
create_comparison_mosaic(os.path.join(os.path.dirname(output_dir), "wild_comparisons"),
                         output_path="/home/sj/working_dir/3m/wild_comparison_mosaic.png")
print("Wild comparison mosaic created and saved as wild_comparison_mosaic.png")


Mosaic saved to: /home/sj/working_dir/3m/wild_comparison_mosaic.png
Wild comparison mosaic created and saved as wild_comparison_mosaic.png
